# 07 — SD4ft-Miner: Úloha 1
## Ženy vs. muži dle věkové skupiny
### 4IZ503 Projektový seminář — Ultra Marathon Running

---

### Slovní zadání

Ve kterých věkových skupinách jsou ženy relativně rychlejší než muži
(mají vyšší pravděpodobnost být v rychlé třetině závodního pole)?

**Hypotéza:** Gender gap se zmenšuje s věkem. V kategorii 50+ by ženy
měly mít relativně lepší výsledky vůči mužům než v kategorii 18-29,
protože vytrvalost a zkušenost (ženské výhody) jsou důležitější
ve vyšším věku, zatímco absolutní rychlost (mužská výhoda) klesá.

**Ante:** `age_group(subset)`, maxlen=1  
**Succ:** `speed_cat(rychlý)`  
**Frst:** `gender(F)` — ženy  
**Scnd:** `gender(M)` — muži  

SD4ft-Miner porovnává `conf(Frst & Ante ⟹ Succ)` vs `conf(Scnd & Ante ⟹ Succ)`
— tzn. v jaké věkové skupině jsou ženy relativně úspěšnější než muži.

---

### Parametry úlohy

| Parametr | Hodnota |
|---|---|
| Procedura | SD4ft-Miner |
| FrstBase (min. počet záznamů — ženy) | 50 |
| ScndBase (min. počet záznamů — muži) | 50 |
| Ratioconf (min. poměr confidence F/M) | 1.2 |
| Ante | age_group(subset), maxlen=1 |
| Succ | speed_cat(rychlý) |
| Frst | gender(F) |
| Scnd | gender(M) |
| Data | ultra_clean_cm.parquet (~6.87M záznamů) |

> ⚠️ **Metodická poznámka:** Ratioconf = conf_frst / conf_scnd ≥ 1.2
> znamená, že ženy mají alespoň o 20 % vyšší pravděpodobnost být
> v rychlé třetině než muži ve stejné věkové skupině.

## 1. Import a načtení dat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from cleverminer import cleverminer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')

df_cm = pd.read_parquet(DATA_DIR / 'ultra_clean_cm.parquet')
print(f"Načteno: {len(df_cm):,} řádků")
print(f"Sloupce: {df_cm.columns.tolist()}")

## 2. Příprava dat pro úlohu

In [ ]:
# Pro tuto úlohu potřebujeme: age_group, gender, speed_cat
cols = ['age_group', 'gender', 'speed_cat']
df_task = df_cm[cols].dropna().copy()

print(f"Záznamy s kompletními daty: {len(df_task):,}")
print()
print("Rozložení age_group:")
print(df_task['age_group'].value_counts().sort_index())
print()
print("Rozložení gender:")
print(df_task['gender'].value_counts())
print()
print("Rozložení speed_cat (ověření ~33/33/33):")
print((df_task['speed_cat'].value_counts() / len(df_task) * 100).round(1))
print()
# Křížová tabulka age_group × gender
print("Záznamy dle age_group × gender:")
print(pd.crosstab(df_task['age_group'], df_task['gender']))

## 3. CleverMiner — SD4ft-Miner úloha

In [ ]:
cm = cleverminer(df=df_task)

cm.mine(
    proc='SD4ftMiner',
    quantifiers={'FrstBase': 50, 'ScndBase': 50, 'Ratioconf': 1.2},
    ante={
        'attributes': [
            {'name': 'age_group', 'type': 'subset', 'minlen': 1, 'maxlen': 1}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    },
    succ={
        'attributes': [
            {'name': 'speed_cat', 'type': 'one', 'value': 'rychlý'}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    },
    frst={
        'attributes': [{'name': 'gender', 'type': 'one', 'value': 'F'}],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    },
    scnd={
        'attributes': [{'name': 'gender', 'type': 'one', 'value': 'M'}],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    }
)

print("\nSouhrn:")
cm.print_summary()

## 4. Výsledky

In [ ]:
print("Všechna pravidla (seřazená dle Ratioconf):")
cm.print_rulelist(sortby='ratioconf', storesorted=True)

## 5. Extrakce pravidel pro analýzu

In [ ]:
rules = []
n = cm.get_rulecount()

for i in range(1, n + 1):
    quant = cm.get_quantifiers(i)
    rule_text = cm.get_ruletext(i)

    # Parsování věkové skupiny z textu pravidla
    age_match = re.search(r'age_group\(([^)]+)\)', rule_text)

    rules.append({
        'rule_id':    i,
        'age_group':  age_match.group(1) if age_match else None,
        'frstbase':   quant.get('frstbase'),
        'scndbase':   quant.get('scndbase'),
        'frstconf':   quant.get('frstconf'),
        'scndconf':   quant.get('scndconf'),
        'ratioconf':  quant.get('ratioconf'),
        'rule_text':  rule_text,
    })

df_rules = pd.DataFrame(rules)
print(f"Extrahováno {len(df_rules)} pravidel")
print()
print(df_rules.sort_values('ratioconf', ascending=False).to_string(index=False))

## 6. Vizualizace

In [ ]:
age_order = ['18-29', '30-39', '40-49', '50-59', '60-69', '70+']

# Doplníme confidence pro všechny věkové skupiny přímým výpočtem z dat
# (i pro skupiny kde SD4ft nenalezl pravidla — pro kontext)
baseline_data = []
for age in age_order:
    for gender in ['F', 'M']:
        subset = df_task[(df_task['age_group'] == age) & (df_task['gender'] == gender)]
        if len(subset) > 0:
            conf = (subset['speed_cat'] == 'rychlý').mean()
            baseline_data.append({'age_group': age, 'gender': gender,
                                   'conf': conf, 'n': len(subset)})

df_base = pd.DataFrame(baseline_data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('SD4ft-Miner: Ženy vs. muži dle věkové skupiny\n'
             '(succ: speed_cat = rychlý, Ratioconf F/M ≥ 1.2)',
             fontsize=13, fontweight='bold')

# Graf 1 — Confidence pro F a M dle věkové skupiny (z dat)
ages_present = [a for a in age_order if a in df_base['age_group'].values]
x = np.arange(len(ages_present))
width = 0.35

df_f = df_base[df_base['gender'] == 'F'].set_index('age_group')['conf']
df_m = df_base[df_base['gender'] == 'M'].set_index('age_group')['conf']

bars_f = ax1.bar(x - width/2,
                 [df_f.get(a, np.nan) for a in ages_present],
                 width, label='Ženy (F)', color='tomato', alpha=0.85, edgecolor='white')
bars_m = ax1.bar(x + width/2,
                 [df_m.get(a, np.nan) for a in ages_present],
                 width, label='Muži (M)', color='steelblue', alpha=0.85, edgecolor='white')

ax1.axhline(0.333, color='black', linewidth=1, linestyle='--', label='Průměr (33%)')

for bars in [bars_f, bars_m]:
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h) and h > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=7)

ax1.set_xlabel('Věková skupina', fontsize=11)
ax1.set_ylabel('Confidence (podíl rychlých)', fontsize=11)
ax1.set_title('Podíl rychlých závodníků dle pohlaví a věku', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels(ages_present, fontsize=9)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0.25, 0.45)

# Graf 2 — Ratioconf (F/M) z SD4ft pravidel + baseline z dat
# Výpočet ratioconf z baseline dat pro všechny věkové skupiny
ratio_data = []
for age in ages_present:
    f_conf = df_f.get(age, np.nan)
    m_conf = df_m.get(age, np.nan)
    if not np.isnan(f_conf) and not np.isnan(m_conf) and m_conf > 0:
        ratio_data.append({'age_group': age, 'ratioconf': f_conf / m_conf})

df_ratio = pd.DataFrame(ratio_data)
colors_ratio = ['tomato' if r >= 1.2 else 'steelblue'
                for r in df_ratio['ratioconf'].values]

bars2 = ax2.bar(np.arange(len(df_ratio)), df_ratio['ratioconf'],
                color=colors_ratio, alpha=0.85, edgecolor='white')
for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.005,
             f'{h:.3f}', ha='center', va='bottom', fontsize=9)

ax2.axhline(1.0,  color='black', linewidth=1, linestyle='--', label='Parití F=M (1.0)')
ax2.axhline(1.2,  color='red',   linewidth=1, linestyle=':',  label='Práh Ratioconf=1.2')

import matplotlib.patches as mpatches
patch_f = mpatches.Patch(color='tomato',    alpha=0.85, label='F > práh 1.2')
patch_m = mpatches.Patch(color='steelblue', alpha=0.85, label='F < práh 1.2')
ax2.legend(handles=[patch_f, patch_m,
                    plt.Line2D([0],[0], color='black', linestyle='--'),
                    plt.Line2D([0],[0], color='red',   linestyle=':')],
           labels=['F > práh 1.2', 'F < práh 1.2', 'Parita F=M (1.0)', 'Práh Ratioconf=1.2'],
           fontsize=9)

ax2.set_xlabel('Věková skupina', fontsize=11)
ax2.set_ylabel('Ratioconf (conf_F / conf_M)', fontsize=11)
ax2.set_title('Relativní výhoda žen vůči mužům dle věku\n'
              '(červené sloupce = ženy splňují SD4ft práh)', fontsize=11)
ax2.set_xticks(np.arange(len(df_ratio)))
ax2.set_xticklabels(df_ratio['age_group'].tolist(), fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / '07_zeny_vek.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graf uložen.")

## 7. Interpretace grafů

**Graf vlevo — Confidence (podíl rychlých) dle pohlaví a věku:**

Ukazuje absolutní výkonnost v rámci závodního pole. Muži mají
obecně vyšší confidence (jsou častěji v rychlé třetině). Sledujeme,
jak se tento rozdíl mění s věkem.

**Graf vpravo — Ratioconf (F/M) dle věkové skupiny:**

Ratioconf = conf_F / conf_M > 1 znamená, že ženy jsou v dané
věkové skupině relativně úspěšnější. Červené sloupce splňují
SD4ft práh ≥ 1.2 — ženy jsou o 20+ % úspěšnější než muži.
Trend roste-li s věkem, hypotéza je potvrzena.

## 8. Zajímavá pravidla

In [ ]:
print("=== ZAJÍMAVÁ PRAVIDLA ===")
print()

if len(df_rules) > 0:
    df_sorted = df_rules.sort_values('ratioconf', ascending=False)
    print("TOP 3 pravidla (nejvyšší Ratioconf — ženy nejvíce dominují):")
    for _, row in df_sorted.head(3).iterrows():
        cm.print_rule(int(row['rule_id']))
        print()
else:
    print("SD4ft-Miner nenalezl pravidla splňující Ratioconf ≥ 1.2.")
    print("Ženy nedosahují o 20+ % vyšší confidence než muži v žádné věkové skupině.")
    print()
    print("Baseline Ratioconf z dat:")
    print(df_ratio.sort_values('ratioconf', ascending=False).to_string(index=False))

## 9. Souhrn a business interpretace

In [ ]:
print("=" * 60)
print("SOUHRN — Úloha 1 (SD4ft): Ženy vs. muži dle věkové skupiny")
print("=" * 60)
print()

print(f"Celkem nalezených SD4ft pravidel (Ratioconf ≥ 1.2): {len(df_rules)}")
print()

if len(df_ratio) > 0:
    max_ratio = df_ratio.sort_values('ratioconf', ascending=False).iloc[0]
    min_ratio = df_ratio.sort_values('ratioconf').iloc[0]
    print(f"Nejvyšší Ratioconf: {max_ratio['age_group']} = {max_ratio['ratioconf']:.3f}")
    print(f"Nejnižší Ratioconf: {min_ratio['age_group']} = {min_ratio['ratioconf']:.3f}")
    print()

    # Trend — korelace věku s ratioconf
    age_numeric = {'18-29': 1, '30-39': 2, '40-49': 3, '50-59': 4, '60-69': 5, '70+': 6}
    df_ratio['age_num'] = df_ratio['age_group'].map(age_numeric)
    corr = df_ratio['age_num'].corr(df_ratio['ratioconf'])
    print(f"Korelace věk vs Ratioconf: {corr:.3f}")
    if corr > 0.3:
        print("→ Pozitivní korelace — relativní výhoda žen ROSTE s věkem ✓")
    elif corr < -0.3:
        print("→ Negativní korelace — relativní výhoda žen KLESÁ s věkem")
    else:
        print("→ Žádný jasný trend — věk neovlivňuje gender gap")

print()
print("BUSINESS DOPORUČENÍ:")
print("  → Pokud ženy dominují ve vyšším věku: marketing ultra závodů pro ženy 40+")
print("  → Věkově specifické ženské kategorie pro lepší viditelnost výsledků")
print("  → Trenéři: strategie pro ženy 50+ může kopírovat mužské přístupy")

## Shrnutí

**Metoda:** SD4ft-Miner (CleverMiner 1.2.6). Porovnává pravidla
`age_group(X) ⟹ speed_cat(rychlý)` pro ženy (Frst) vs. muže (Scnd).
Pravidlo platí pokud conf_F / conf_M ≥ 1.2.

**Data:** ~6.87M závodníků, speed_cat per event.

**Klíčový nález:** Analýza Ratioconf dle věkové skupiny ukazuje,
v jakém věku mají ženy relativně nejlepší výsledky vůči mužům.

**Limitace:**
- speed_cat per event — srovnáváme relativní výkonnost, ne absolutní časy
- Dataset má 19.8 % žen — menší statistická síla zejména u 70+
- SD4ft neporovnává absolutní výkonnost, jen relativní podíl rychlých

**Další notebook:** `08_SD4ft_uloha2.ipynb` — Zkušenosti dle vzdálenosti (SD4ft-Miner)